<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drexelwireless/ece512-courseware/blob/main/notebooks/lectures/week08-diversity.ipynb)

# Week 8 — Diversity and MIMO
**ECE 512 Wireless Communications** · companion notebook to the Week 8 lecture

This notebook lets you experiment with the ideas from the slides:

1. Diversity branches: time, frequency, spatial and polarization diversity
2. Selection diversity: $P_M(\gamma) = \left(1 - e^{-\gamma/\Gamma}\right)^M$
3. Maximal ratio combining: $\gamma_{\mathrm{OUTPUT}} = \sum_i \gamma_i$
4. Equal gain combining, and SC vs. EGC vs. MRC
5. Outage probability vs. number of branches $M$
6. BPSK bit error rate with diversity: diversity order as a slope
7. MIMO space-time coding: Alamouti $2\times 1$ vs. $1\times 2$ MRC
8. MIMO capacity teaser: $\log_2\det$ (bridge to Week 9)

Throughout: $M$ diversity branches, each i.i.d. Rayleigh with instantaneous SNR $\gamma_i$ and mean SNR
$\Gamma = \mathrm{E}[\gamma_i]$, per-branch noise power $N$, combiner output SNR $\gamma_{\mathrm{OUTPUT}}$,
and $M_T$ / $M_R$ transmit / receive antennas.

Run the setup cell first. In Colab: *Runtime → Run all*.

In [ ]:
# --- Setup: installs the course package in Google Colab (safe to re-run) ---
COURSEWARE_URL = "git+https://github.com/drexelwireless/ece512-courseware"
import importlib.util
if importlib.util.find_spec("ece512") is None:
    get_ipython().run_line_magic("pip", f"install -q {COURSEWARE_URL}")

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
from ece512 import diversity as dv
from ece512 import fading, modulation as mod

rng = np.random.default_rng(8)
db = lambda x: 10 * np.log10(x)
lin = lambda x_db: 10 ** (np.asarray(x_db) / 10)
# one fixed color per combiner, used in every figure
COLOR = {"SC": "tab:blue", "EGC": "tab:orange", "MRC": "tab:green", "none": "gray"}

## 1. Diversity branches
Diversity works only if the branches fade **independently** (or are highly uncorrelated), so that they are
unlikely to be in a deep fade at the same time. Each technique buys independence with a different resource:

| Technique | Branches separated by more than … | Complex-gain correlation (model used here) |
|---|---|---|
| Time | the coherence time $T_c$ | $\rho = J_0(2\pi f_m \tau)$ |
| Frequency | the coherence bandwidth $B_c$ | $\lvert\rho\rvert = 1/\sqrt{1 + (2\pi\,\Delta f\,\sigma_\tau)^2}$ (exponential PDP) |
| Spatial | the coherence distance | $\rho = J_0(2\pi d/\lambda)$ (Clarke) |
| Polarization | — (co-located, orthogonal polarizations) | $\rho$ set by the scattering; branch 2 weaker by the XPD |

The envelope-squared (power) correlation is $\lvert\rho\rvert^2$. Below, each row shows two branches that are
**correlated** (left) and nearly **independent** (right), for a mobile with $f_m = 50$ Hz. The thick gray band is what
a selection combiner would output (the stronger of the two).

In [ ]:
fm, fs = 50.0, 2000.0
t = np.arange(0, 0.5, 1 / fs)
sig_tau = 1e-6                                   # 1 us rms delay spread -> B_c ~ 160 kHz

def branches(kind, sep, seed=1):
    # returns (h1, h2, theoretical |rho|, label for the separation)
    if kind == "spatial":
        h = dv.spatial_branches(fm, t, [0, sep], rng=seed)
        return h[0], h[1], abs(dv.clarke_correlation(sep)), rf"$d = {sep:g}\lambda$"
    if kind == "time":                            # same channel, second copy sent tau later
        tau = sep / 1e3
        h = dv.spatial_branches(fm, t, [0, fm * tau], rng=seed)
        return h[0], h[1], abs(dv.clarke_correlation(fm * tau)), rf"$\tau = {sep:g}$ ms"
    if kind == "frequency":
        df = sep * 1e3
        H = dv.wideband_sos(fm, t, [0, df], sig_tau, rng=seed)
        return H[0], H[1], dv.frequency_correlation(df, sig_tau), rf"$\Delta f = {sep:g}$ kHz"
    if kind == "polarization":
        h = dv.polarization_branches(fm, t, sep, xpd_db=3, rng=seed)
        return h[0], h[1], sep, rf"$\rho = {sep:g}$, XPD = 3 dB"

cases = {"spatial": (0.05, 0.5), "time": (0.5, 10), "frequency": (10, 500), "polarization": (0.95, 0.1)}
fig, axes = plt.subplots(4, 2, figsize=(12, 10), sharex=True, sharey=True)
for row, (kind, seps) in zip(axes, cases.items()):
    for ax, sep, tag in zip(row, seps, ["correlated", "independent"]):
        h1, h2, rho, lab = branches(kind, sep)
        ax.plot(t * 1e3, db(abs(h1) ** 2), lw=1, color="tab:blue", label="branch 1")
        ax.plot(t * 1e3, db(abs(h2) ** 2), lw=1, color="tab:red", label="branch 2")
        ax.plot(t * 1e3, db(np.maximum(abs(h1) ** 2, abs(h2) ** 2)), lw=3.5, color="k", alpha=0.18,
                zorder=0, label="selected (max)")
        ax.set_title(f"{kind} ({tag}): {lab},  $|\\rho|^2 = {rho**2:.2f}$", fontsize=10)
        ax.set_ylim(-35, 10); ax.grid(alpha=0.3)
    row[0].set_ylabel("power $|h|^2$ (dB)")
for ax in axes[-1]:
    ax.set_xlabel("time (ms)")
axes[0, 0].legend(loc="lower left", ncol=3, fontsize=8)
fig.tight_layout(); plt.show()

**Explore:** pick a technique and move the separation. The measured correlation of the complex gains over this
short record (plus two more seeds) is compared with the model; watch how often *both* branches fade below
$-10$ dB together.

In [ ]:
def show_branch_correlation(kind="spatial", separation=0.15):
    rng_sep = {"spatial": (0.0, 2.0, "d/λ"), "time": (0.0, 40.0, "τ (ms)"),
               "frequency": (0.0, 1000.0, "Δf (kHz)"), "polarization": (0.0, 1.0, "ρ")}[kind]
    lo, hi, unit = rng_sep
    sep = lo + (hi - lo) * separation            # slider is a fraction of the range for this technique
    h1, h2, rho_th, lab = branches(kind, sep, seed=3)
    rho_sim = np.mean([abs(np.corrcoef(*branches(kind, sep, seed=s)[:2])[0, 1]) for s in (3, 4, 5)])
    both = np.mean((abs(h1) ** 2 < 0.1) & (abs(h2) ** 2 < 0.1 * np.mean(abs(h2) ** 2)))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw=dict(width_ratios=[2.2, 1]))
    a1.plot(t * 1e3, db(abs(h1) ** 2), lw=1, color="tab:blue", label="branch 1")
    a1.plot(t * 1e3, db(abs(h2) ** 2), lw=1, color="tab:red", label="branch 2")
    a1.axhline(-10, color="gray", ls=":", lw=1)
    a1.set_ylim(-35, 10); a1.set_xlabel("time (ms)"); a1.set_ylabel("power $|h|^2$ (dB)")
    a1.set_title(f"{kind}: {lab}"); a1.legend(loc="lower left", ncol=2); a1.grid(alpha=0.3)
    a2.plot(db(abs(h1) ** 2), db(abs(h2) ** 2), ".", ms=2, alpha=0.4, color="tab:purple")
    a2.set_xlim(-35, 10); a2.set_ylim(-35, 10); a2.set_aspect("equal")
    a2.set_xlabel("branch 1 (dB)"); a2.set_ylabel("branch 2 (dB)"); a2.grid(alpha=0.3)
    a2.set_title(f"|ρ| model {rho_th:.2f}, measured ≈ {rho_sim:.2f}", fontsize=10)
    fig.tight_layout(); plt.show()
    print(f"fraction of time both branches are > 10 dB below their mean: {both:.3f}")

interact(show_branch_correlation,
         kind=Dropdown(options=["spatial", "time", "frequency", "polarization"], value="spatial"),
         separation=FloatSlider(value=0.15, min=0.0, max=1.0, step=0.01, description="separation"));

*Note:* the "measured" correlation uses only a 0.5 s record (25 Doppler cycles), so it fluctuates around the model.
The first null of $J_0(2\pi d/\lambda)$ is at $d \approx 0.38\lambda$ — the classic "half-wavelength spacing" rule of
thumb for spatial diversity at a mobile in rich scattering.

## 2. Selection diversity
With i.i.d. Rayleigh branches each $\gamma_i$ is exponential, $p(\gamma_i) = \frac{1}{\Gamma}e^{-\gamma_i/\Gamma}$.
Selection fails only if **all** $M$ branches are below the threshold $\gamma$:

$$\Pr[\gamma_1,\ldots,\gamma_M \le \gamma] = \left(1 - e^{-\gamma/\Gamma}\right)^M = P_M(\gamma)$$

The mean output SNR grows only like the harmonic series:
$$\bar\gamma = \Gamma \sum_{k=1}^{M}\frac{1}{k}.$$

Below: Monte Carlo (dots, $2\times10^5$ trials) vs. $P_M(\gamma)$ (lines). Plotting the CDF on a log axis shows the
key effect: the probability of a deep fade falls like $(\gamma/\Gamma)^M$.

In [ ]:
Gamma = 1.0                                  # results depend only on gamma/Gamma
n_mc = 200_000
x_db = np.linspace(-40, 10, 60)              # gamma/Gamma in dB
fig, ax = plt.subplots(figsize=(7.5, 5))
for m, c in zip([1, 2, 3, 4, 6], plt.cm.viridis(np.linspace(0, 0.85, 5))):
    g_sc = dv.selection_combine(dv.branch_snrs(n_mc, m, Gamma, rng))
    sim = np.searchsorted(np.sort(g_sc), lin(x_db) * Gamma) / n_mc
    ax.semilogy(x_db, dv.sc_cdf(lin(x_db) * Gamma, Gamma, m), color=c, label=f"M = {m}")
    ok = sim > 20 / n_mc                     # hide points with too few samples
    ax.semilogy(x_db[ok][::2], sim[ok][::2], "o", ms=4, mfc="none", color=c)
ax.set_ylim(1e-4, 1.1); ax.set_xlim(-40, 10)
ax.set_xlabel(r"$\gamma/\Gamma$ (dB)"); ax.set_ylabel(r"$\Pr[\gamma_{\mathrm{OUTPUT}} \leq \gamma] = P_M(\gamma)$")
ax.set_title("Selection diversity: output SNR CDF (lines: theory, circles: simulation)")
ax.grid(True, which="both", alpha=0.3); ax.legend(title="branches"); plt.show()

In [ ]:
print(f"{'M':>3} {'mean SNR / Gamma (sim)':>24} {'sum 1/k':>9} {'gain (dB)':>10}")
for m in [1, 2, 3, 4, 6, 8, 16]:
    sim = dv.selection_combine(dv.branch_snrs(100_000, m, Gamma, rng)).mean()
    th = dv.sc_mean_snr(Gamma, m)
    print(f"{m:>3} {sim:24.3f} {th:9.3f} {db(th):10.2f}")

Going from 1 to 2 branches adds 1.76 dB of *mean* SNR, but every further branch adds less (16 branches: only 5.3 dB).
The real benefit of selection diversity is not the mean — it is the lower tail of the CDF (fewer deep fades).

## 3. Maximal ratio combining
The combiner output is $r_{\mathrm{COMB}} = \sum_i G_i r_i$ and the noise power is $N_{\mathrm{tot}} = N\sum_i G_i^2$, so
$$\gamma_{\mathrm{OUTPUT}} = \frac{\left(\sum_{i=1}^M G_i r_i\right)^2}{2N\sum_{i=1}^M G_i^2} \le \sum_{i=1}^M \gamma_i,$$
with equality (Cauchy–Schwarz) when $G_i \propto r_i$ (in complex baseband, $G_i \propto h_i^*$). The output SNR is a sum
of $M$ exponentials — chi-square with $2M$ degrees of freedom (Erlang):
$$p(\gamma_{\mathrm{OUTPUT}}) = \frac{\gamma_{\mathrm{OUTPUT}}^{M-1} e^{-\gamma_{\mathrm{OUTPUT}}/\Gamma}}{\Gamma^M (M-1)!},\qquad
\Pr(\gamma_{\mathrm{OUTPUT}} \le \gamma) = 1 - e^{-\gamma/\Gamma}\sum_{k=1}^{M}\frac{(\gamma/\Gamma)^{k-1}}{(k-1)!},\qquad
\bar\gamma = M\Gamma.$$

First a quick check that the *weights* matter: we simulate the signals $r_i = h_i s + n_i$ and compare the SNR of
$\sum_i G_i r_i$ for random weights, co-phased equal weights, and $G_i = h_i^*$.

In [ ]:
M_demo, Gamma_demo, n = 4, 1.0, 20_000
h = fading.iid_rayleigh((n, M_demo), rng)            # E|h|^2 = 1, per-branch noise power N = 1/Gamma
N = 1 / Gamma_demo
gamma_i = abs(h) ** 2 / N
def out_snr(G):                                       # |sum G h|^2 / (N sum |G|^2)
    return abs(np.sum(G * h, axis=1)) ** 2 / (N * np.sum(abs(G) ** 2, axis=1))
rows = {"random complex G_i": out_snr(fading.iid_rayleigh((n, M_demo), rng)),
        "co-phased, |G_i| = 1 (EGC)": out_snr(np.exp(-1j * np.angle(h))),
        "G_i = h_i^* (MRC)": out_snr(np.conj(h)),
        "bound: sum gamma_i": gamma_i.sum(axis=1)}
for k, v in rows.items():
    print(f"{k:>28}: mean gamma_OUTPUT = {v.mean():.3f} (M Gamma = {M_demo * Gamma_demo:.0f})")

In [ ]:
def show_mrc(M=3):
    Gamma = 1.0
    g_mrc = dv.mrc_combine(dv.branch_snrs(100_000, M, Gamma, rng))
    g_sc = dv.selection_combine(dv.branch_snrs(100_000, M, Gamma, rng))
    x = np.linspace(0, 3 * M + 5, 400)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
    a1.hist(g_mrc, bins=80, range=(0, x[-1]), density=True, alpha=0.35, color=COLOR["MRC"], label="MRC simulation")
    a1.plot(x, dv.mrc_pdf(x, Gamma, M), color=COLOR["MRC"], label="MRC theory (Erlang)")
    a1.hist(g_sc, bins=80, range=(0, x[-1]), density=True, alpha=0.35, color=COLOR["SC"], label="SC simulation")
    a1.plot(x, dv.sc_pdf(x, Gamma, M), color=COLOR["SC"], label="SC theory")
    a1.axvline(dv.mrc_mean_snr(Gamma, M), color=COLOR["MRC"], ls="--", lw=1)
    a1.axvline(dv.sc_mean_snr(Gamma, M), color=COLOR["SC"], ls="--", lw=1)
    a1.set_xlabel(r"$\gamma_{\mathrm{OUTPUT}}/\Gamma$"); a1.set_ylabel("probability density")
    a1.set_title(f"Output SNR density, M = {M} (dashed: means)"); a1.legend(fontsize=8); a1.grid(alpha=0.3)
    xd = np.linspace(-30, 15, 200)
    for name, cdf, sim in [("MRC", dv.mrc_cdf, g_mrc), ("SC", dv.sc_cdf, g_sc)]:
        a2.semilogy(xd, cdf(lin(xd), Gamma, M), color=COLOR[name], label=f"{name} theory")
        xs = xd[::8]; emp = np.searchsorted(np.sort(sim), lin(xs)) / sim.size
        ok = emp > 1e-4
        a2.semilogy(xs[ok], emp[ok], "o", mfc="none", color=COLOR[name], label=f"{name} simulation")
    a2.semilogy(xd, dv.sc_cdf(lin(xd), Gamma, 1), color=COLOR["none"], ls=":", label="no diversity (M = 1)")
    a2.set_ylim(1e-4, 1.1); a2.set_xlabel(r"$\gamma/\Gamma$ (dB)")
    a2.set_ylabel(r"$\Pr(\gamma_{\mathrm{OUTPUT}} \leq \gamma)$"); a2.set_title("Output SNR CDF")
    a2.legend(fontsize=8); a2.grid(True, which="both", alpha=0.3)
    fig.tight_layout(); plt.show()
    print(f"mean SNR gain: MRC {db(dv.mrc_mean_snr(1, M)):.2f} dB (M Gamma) vs. SC {db(dv.sc_mean_snr(1, M)):.2f} dB")

interact(show_mrc, M=IntSlider(value=3, min=1, max=8, step=1));

## 4. Equal gain combining
EGC co-phases the branches but uses unit gains, so it needs phase (not amplitude) estimates:
$$\gamma_{\mathrm{OUTPUT}} = \frac{\left(\sum_{i=1}^M \sqrt{\gamma_i}\right)^2}{M}.$$
For Rayleigh branches $\mathrm{E}[\sqrt{\gamma_i}] = \sqrt{\pi\Gamma}/2$, which gives the mean
$\bar\gamma_{\mathrm{EGC}} = \Gamma\left[1 + (M-1)\frac{\pi}{4}\right]$ — within about 1 dB of $M\Gamma$ for any $M$.

EGC is better than SC *on average*, but not in every realization: if one branch is strong and the others are in deep
fades, adding the weak branches with full (unit) gain adds more noise than signal. MRC, by contrast, is never worse than SC.

In [ ]:
Ms = np.arange(1, 17)
sim = {k: [] for k in ("SC", "EGC", "MRC")}
lose = []
for m in Ms:
    g = dv.branch_snrs(40_000, m, 1.0, rng)
    sim["SC"].append(dv.selection_combine(g).mean()); sim["EGC"].append(dv.egc_combine(g).mean())
    sim["MRC"].append(dv.mrc_combine(g).mean())
    lose.append(np.mean(dv.egc_combine(g) < dv.selection_combine(g)))
theory = {"SC": [dv.sc_mean_snr(1, m) for m in Ms], "EGC": dv.egc_mean_snr(1, Ms), "MRC": dv.mrc_mean_snr(1, Ms)}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
for k in sim:
    a1.plot(Ms, db(np.array(theory[k])), color=COLOR[k], label=f"{k} theory")
    a1.plot(Ms, db(np.array(sim[k])), "o", mfc="none", color=COLOR[k], label=f"{k} simulation")
a1.set_xlabel("number of branches M"); a1.set_ylabel(r"mean SNR gain $\bar\gamma/\Gamma$ (dB)")
a1.set_title("Mean output SNR"); a1.legend(fontsize=8, ncol=2); a1.grid(alpha=0.3)

m = 4
g = dv.branch_snrs(100_000, m, 1.0, rng)
xd = np.linspace(-25, 15, 200)
for k, f in [("SC", dv.selection_combine), ("EGC", dv.egc_combine), ("MRC", dv.mrc_combine)]:
    a2.semilogy(xd, np.searchsorted(np.sort(f(g)), lin(xd)) / g.shape[0], color=COLOR[k], label=k)
a2.semilogy(xd, dv.sc_cdf(lin(xd), 1, 1), color=COLOR["none"], ls=":", label="M = 1")
a2.set_ylim(1e-4, 1.1); a2.set_xlabel(r"$\gamma/\Gamma$ (dB)"); a2.set_ylabel(r"$\Pr(\gamma_{\mathrm{OUTPUT}} \leq \gamma)$")
a2.set_title(f"Output SNR CDF, M = {m} (simulation)"); a2.legend(); a2.grid(True, which="both", alpha=0.3)
fig.tight_layout(); plt.show()
print(f"MRC - EGC gap at M = 16: {db(theory['MRC'][-1] / theory['EGC'][-1]):.2f} dB;  "
      f"fraction of realizations where EGC < SC (M = 4): {lose[3]:.3f}")

## 5. Outage probability vs. $M$
A link is in **outage** when $\gamma_{\mathrm{OUTPUT}}$ falls below the threshold $\gamma_{\mathrm{th}}$ needed by the receiver:
$P_{\mathrm{out}} = \Pr(\gamma_{\mathrm{OUTPUT}} < \gamma_{\mathrm{th}})$ — exactly the CDFs above evaluated at $\gamma_{\mathrm{th}}$.
Selection and MRC have closed forms; EGC is simulated. Move the fade margin $\Gamma/\gamma_{\mathrm{th}}$ and see how many branches
you need for a 1% or 0.1% outage target.

In [ ]:
def show_outage(margin_db=10.0):
    Ms = np.arange(1, 9)
    gth = lin(-margin_db)                      # gamma_th / Gamma
    egc = [np.mean(dv.egc_combine(dv.branch_snrs(200_000, m, 1.0, rng)) < gth) for m in Ms]
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.semilogy(Ms, dv.outage_probability(gth, 1.0, Ms, "sc"), "o-", color=COLOR["SC"], label="SC (theory)")
    egc = np.array(egc); ok = egc > 0
    ax.semilogy(Ms[ok], egc[ok], "s--", color=COLOR["EGC"], label="EGC (simulation, 2×10$ trials)")
    ax.semilogy(Ms, dv.outage_probability(gth, 1.0, Ms, "mrc"), "^-", color=COLOR["MRC"], label="MRC (theory)")
    for p in (1e-2, 1e-3):
        ax.axhline(p, color="gray", ls=":", lw=1)
    ax.set_ylim(1e-8, 1); ax.set_xlabel("number of branches M")
    ax.set_ylabel(r"$P_{\mathrm{out}} = \Pr(\gamma_{\mathrm{OUTPUT}} < \gamma_{\mathrm{th}})$")
    ax.set_title(rf"Outage vs. M, fade margin $\Gamma/\gamma_{{\mathrm{{th}}}}$ = {margin_db:.0f} dB")
    ax.grid(True, which="both", alpha=0.3); ax.legend(); plt.show()
    for name in ("sc", "mrc"):
        need = [m for m in range(1, 50) if dv.outage_probability(gth, 1.0, m, name) < 1e-3]
        print(f"{name.upper():>3}: branches needed for P_out < 0.1%: {need[0] if need else '> 49'}")

interact(show_outage, margin_db=FloatSlider(value=10, min=0, max=25, step=1, description="margin (dB)"));

For a single Rayleigh branch, $P_{\mathrm{out}} \approx \gamma_{\mathrm{th}}/\Gamma$: 1% outage needs a 20 dB fade margin.
With two branches the same target needs only about 10 dB (SC) — diversity trades hardware for fade margin.

## 6. BPSK bit error rate with diversity
Averaging $Q\!\left(\sqrt{2\gamma_{\mathrm{OUTPUT}}}\right)$ over the output SNR density gives (with $\Gamma = E_b/N_0$ per branch,
$\mu = \sqrt{\Gamma/(1+\Gamma)}$):

* **MRC:** $P_b = \left(\frac{1-\mu}{2}\right)^M \sum_{k=0}^{M-1}\binom{M-1+k}{k}\left(\frac{1+\mu}{2}\right)^k \;\approx\; \binom{2M-1}{M}\left(4\Gamma\right)^{-M}$
* **SC:** $P_b = \frac12\sum_{k=0}^{M}(-1)^k\binom{M}{k}\left(1 + k/\Gamma\right)^{-1/2}$ (evaluated numerically)
* **EGC, $M = 2$:** $P_b = \frac12\left[1 - \sqrt{1 - (1+\Gamma)^{-2}}\right]$; simulation only for $M > 2$

At high SNR every scheme has $P_b \propto \Gamma^{-M}$: on a log–log plot the BER curve has slope $-M$ decades per
decade of SNR. That slope is the **diversity order**; the combiner choice only shifts the curve horizontally (array/combining gain).

The Monte Carlo transmits BPSK symbols through complex Rayleigh gains $h_i$ and adds noise of power $N = N_0$ on each
branch; the combiners use the true $h_i$ (perfect channel knowledge).

In [ ]:
snr_db = np.arange(0, 31, 1.0)
snr_mc = np.arange(0, 26, 2.5)

def show_ber(M=2, n_bits=100_000):
    G = lin(snr_db)
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.semilogy(snr_db, mod.ber_bpsk_awgn(snr_db), color="k", lw=1, label="AWGN (no fading)")
    ax.semilogy(snr_db, mod.ber_bpsk_rayleigh(snr_db), color=COLOR["none"], ls=":", label="Rayleigh, M = 1")
    theory = {"SC": dv.ber_bpsk_sc(G, M), "MRC": dv.ber_bpsk_mrc(G, M)}
    if M <= 2:
        theory["EGC"] = dv.ber_bpsk_egc2(G) if M == 2 else dv.ber_bpsk_mrc(G, 1)
    for k in ("SC", "EGC", "MRC"):
        if k in theory:
            ax.semilogy(snr_db, theory[k], color=COLOR[k], label=f"{k} theory")
        sim = dv.simulate_ber_bpsk(snr_mc, M, k.lower(), n_bits=n_bits, rng=rng)
        ok = sim * n_bits >= 10                 # keep points with at least 10 errors
        ax.semilogy(snr_mc[ok], sim[ok], "o", mfc="none", ms=7, color=COLOR[k], label=f"{k} simulation")
    # slope guide: a line falling M decades per decade of SNR, a decade below the MRC curve
    s0 = snr_db[np.argmax(dv.ber_bpsk_mrc(G, M) < 1e-3)]      # where MRC crosses 1e-3
    xs = np.array([s0, min(s0 + 30 / M, 30)]); ys = 1e-4 * lin(s0 - xs) ** M
    ax.semilogy(xs, ys, "k--", lw=1)
    ax.annotate(f"slope −{M} decades/decade\n= diversity order {M}", (xs[1], ys[1]),
                xytext=(-4, -4), textcoords="offset points", ha="right", va="top", fontsize=9,
                bbox=dict(boxstyle="round", fc="white", ec="none", alpha=0.85))
    ax.set_ylim(1e-8, 0.5); ax.set_xlim(0, 30)
    ax.set_xlabel(r"mean SNR per branch $\Gamma = E_b/N_0$ (dB)"); ax.set_ylabel("bit error rate $P_b$")
    ax.set_title(f"Coherent BPSK, M = {M} i.i.d. Rayleigh branches")
    ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1)); fig.tight_layout(); plt.show()
    hi = snr_db >= 20
    print(f"fitted diversity order (20–30 dB): SC {dv.diversity_order(snr_db[hi], theory['SC'][hi]):.2f}, "
          f"MRC {dv.diversity_order(snr_db[hi], theory['MRC'][hi]):.2f}")

interact(show_ber, M=IntSlider(value=2, min=1, max=4, step=1),
         n_bits=IntSlider(value=100_000, min=20_000, max=400_000, step=20_000));

Static overview for $M = 1, 2, 4$ (closed forms): the curves fan out with slopes $-1, -2, -4$. At $P_b = 10^{-4}$, going
from 1 to 2 MRC branches saves about 18 dB of mean SNR — far more than the 3 dB increase in mean SNR ($M\Gamma$) would suggest.

In [ ]:
G = lin(snr_db)
fig, ax = plt.subplots(figsize=(8, 5.5))
styles = {1: ":", 2: "--", 4: "-"}
ax.semilogy(snr_db, dv.ber_bpsk_mrc(G, 1), color=COLOR["none"], ls=":", label="M = 1 (SC = MRC)")
for M in (2, 4):
    ax.semilogy(snr_db, dv.ber_bpsk_sc(G, M), color=COLOR["SC"], ls=styles[M], label=f"SC, M = {M}")
    ax.semilogy(snr_db, dv.ber_bpsk_mrc(G, M), color=COLOR["MRC"], ls=styles[M], label=f"MRC, M = {M}")
ax.semilogy(snr_db, mod.ber_bpsk_awgn(snr_db), color="k", lw=1, label="AWGN")
ax.axhline(1e-4, color="gray", lw=0.8)
ax.set_ylim(1e-8, 0.5); ax.set_xlabel(r"$\Gamma = E_b/N_0$ per branch (dB)"); ax.set_ylabel("bit error rate $P_b$")
ax.set_title("BPSK: selection vs. maximal ratio combining")
ax.grid(True, which="both", alpha=0.3); ax.legend(ncol=2, fontsize=8); plt.show()

from scipy.optimize import brentq
req = lambda f: brentq(lambda s: np.log10(f(lin(s))) + 4, -5, 60)
print("Gamma needed for P_b = 1e-4:")
for M in (1, 2, 4):
    print(f"  M = {M}:  SC {req(lambda g: dv.ber_bpsk_sc(g, M)):5.1f} dB   MRC {req(lambda g: dv.ber_bpsk_mrc(g, M)):5.1f} dB")

## 7. MIMO space-time coding: Alamouti $2\times1$ vs. $1\times2$ MRC
Transmit diversity without channel knowledge at the transmitter: Alamouti sends two symbols over two time slots,

| | antenna 1 | antenna 2 |
|---|---|---|
| slot 1 | $s_1/\sqrt2$ | $s_2/\sqrt2$ |
| slot 2 | $-s_2^*/\sqrt2$ | $s_1^*/\sqrt2$ |

With $r_1 = (h_1 s_1 + h_2 s_2)/\sqrt2 + n_1$ and $r_2 = (-h_1 s_2^* + h_2 s_1^*)/\sqrt2 + n_2$, the linear combiner
$\hat s_1 = h_1^* r_1 + h_2 r_2^*$, $\hat s_2 = h_2^* r_1 - h_1 r_2^*$ gives $\frac{|h_1|^2 + |h_2|^2}{\sqrt2}\,s + \text{noise}$:
the **same diversity order (2)** as $1\times2$ MRC, but with $\gamma_{\mathrm{OUTPUT}} = \frac{\Gamma}{2}\left(|h_1|^2 + |h_2|^2\right)$.
The transmitter splits its fixed total power over two antennas, whereas the two receive antennas of MRC each collect the full
power — so Alamouti is exactly **3 dB** behind $1\times2$ MRC.

In [ ]:
snr_a = np.arange(0, 26, 2.5)
n_bits = 200_000
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.semilogy(snr_db, mod.ber_bpsk_rayleigh(snr_db), color=COLOR["none"], ls=":", label="SISO (1×1)")
ax.semilogy(snr_db, dv.ber_bpsk_mrc(G, 2), color=COLOR["MRC"], label="1×2 MRC theory")
ax.semilogy(snr_db, dv.ber_bpsk_alamouti(G), color="tab:purple", label="2×1 Alamouti theory")
sim_mrc = dv.simulate_ber_bpsk(snr_a, 2, "mrc", n_bits=n_bits, rng=rng)
sim_ala = dv.simulate_alamouti_bpsk(snr_a, n_bits=n_bits, rng=rng)
for sim, c, lab in [(sim_mrc, COLOR["MRC"], "1×2 MRC simulation"), (sim_ala, "tab:purple", "2×1 Alamouti simulation")]:
    ok = sim * n_bits >= 10
    ax.semilogy(snr_a[ok], sim[ok], "o", mfc="none", ms=7, color=c, label=lab)
p = 1e-4
s_mrc, s_ala = req(lambda g: dv.ber_bpsk_mrc(g, 2)), req(dv.ber_bpsk_alamouti)
ax.annotate("", xy=(s_ala, p), xytext=(s_mrc, p), arrowprops=dict(arrowstyle="<->", color="k"))
ax.text((s_mrc + s_ala) / 2, p * 1.6, f"{s_ala - s_mrc:.1f} dB", ha="center")
ax.set_ylim(1e-6, 0.5); ax.set_xlim(0, 30)
ax.set_xlabel(r"total transmit $E_b/N_0$ (dB)"); ax.set_ylabel("bit error rate $P_b$")
ax.set_title("Transmit diversity (Alamouti) vs. receive diversity (MRC), BPSK")
ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8, loc="lower left"); plt.show()

## 8. MIMO capacity teaser (bridge to Week 9)
With $\mathbf{Y} = \mathbf{H}\mathbf{X} + \mathbf{N}$ and no channel knowledge at the transmitter (equal power per antenna), the
open-loop capacity of one channel realization is
$$I = \log_2\det\!\left(\mathbf{I}_{M_R} + \frac{E_s}{M_T N_0}\mathbf{H}\mathbf{H}^H\right)
    = \sum_{i=1}^{r}\log_2\!\left(1 + \frac{E_s}{M_T N_0}\lambda_i\right),$$
where $\lambda_i$ are the positive eigenvalues of $\mathbf{H}\mathbf{H}^H$ and $r = \operatorname{rank}\mathbf{H}$. Diversity (Sections 2–7)
uses extra antennas to make the link *reliable*; here they open up to $\min(M_T, M_R)$ parallel spatial "data pipes" so the
ergodic capacity grows **linearly** in $\min(M_T, M_R)$ at high SNR, at the same total power and bandwidth. Week 9 goes deeper.

In [ ]:
def show_capacity(M_T=2, M_R=2):
    s = np.arange(-5, 31, 2.5)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.3))
    for (mt, mr), c in zip([(1, 1), (1, 2), (2, 1), (2, 2), (4, 4)], plt.cm.viridis(np.linspace(0, 0.85, 5))):
        a1.plot(s, dv.ergodic_capacity(mt, mr, s, n_trials=1000, rng=1), color=c, label=f"{mt}×{mr}")
    a1.plot(s, dv.ergodic_capacity(M_T, M_R, s, n_trials=1000, rng=1), "k--", lw=2, label=f"{M_T}×{M_R} (slider)")
    a1.set_xlabel(r"$E_s/N_0$ (dB)"); a1.set_ylabel("ergodic capacity E[I] (bit/s/Hz)")
    a1.set_title(r"Open-loop capacity, i.i.d. Rayleigh $\mathbf{H}$ ($M_T \times M_R$)")
    a1.legend(fontsize=8); a1.grid(alpha=0.3)
    H = fading.iid_rayleigh((3000, M_R, M_T), rng)
    I = dv.mimo_capacity(H, lin(10))
    a2.hist(I, bins=50, density=True, color="tab:purple", alpha=0.6)
    a2.axvline(I.mean(), color="k", ls="--", label=f"mean {I.mean():.2f}")
    a2.axvline(np.quantile(I, 0.1), color="tab:red", ls=":", label=f"10% outage {np.quantile(I, 0.1):.2f}")
    a2.set_xlabel("capacity I (bit/s/Hz)"); a2.set_ylabel("density")
    a2.set_title(f"{M_T}×{M_R} capacity distribution at 10 dB"); a2.legend(); a2.grid(alpha=0.3)
    fig.tight_layout(); plt.show()
    lam = np.linalg.eigvalsh(H[0] @ H[0].conj().T)
    print("one realization: eigenvalues of H H^H =", np.round(np.sort(lam)[::-1], 3))

interact(show_capacity, M_T=IntSlider(value=2, min=1, max=6, step=1), M_R=IntSlider(value=2, min=1, max=6, step=1));

## Try it yourself
* In Section 1, find the smallest antenna spacing $d/\lambda$ for which $\lvert\rho\rvert^2 < 0.1$. A base station sees
  scatterers only within a narrow angle spread — would its required spacing be larger or smaller, and why?
* Selection diversity's mean gain is $\Gamma\sum 1/k$, yet at $P_b = 10^{-4}$ SC with $M = 2$ saves almost as much SNR as MRC.
  Use Sections 2 and 6 to explain why the *mean* SNR is the wrong figure of merit for fading links.
* Modify `dv.simulate_ber_bpsk` inputs (or write your own loop) so that the two branches are correlated,
  $h_2 = \rho h_1 + \sqrt{1-\rho^2}\,h'$. How does the BER slope change as $\rho \to 1$?
* Alamouti is 3 dB behind $1\times2$ MRC. What would a $2\times2$ system using Alamouti at the transmitter and MRC over the two
  receive antennas achieve (diversity order and array gain)? Check your answer with the capacity demo's intuition: when is it
  better to spend antennas on diversity, and when on multiplexing?